In [ ]:
import utils_BiLSTMasTrue
import utils_BiLasTrue_ReLU
import pandas as pd
import matplotlib.pyplot as plt
import utils

In [ ]:
# Load the pickle files
# scaled_train = pd.read_pickle("NIAvalueUpdatedVersion_scaled_train.pkl")
# scaled_test = pd.read_pickle("NIAvalueUpdatedVersion_scaled_test.pkl")
# total_data_p = pd.concat([scaled_train, scaled_test], axis=0)
# Since we want to use the whole scaled decedents data and then we need to read the `NIAvalueUpdatedVersion_scaled_whole_decedents_dataset.pkl` in.
total_data = pd.read_pickle("inputs/NIAvalueUpdatedVersion_scaled_whole_decedents_dataset.pkl")

In [ ]:
total_data # Previous one has the following dataset. 11905 rows x 65 columns.

# Final model for gpath (LSTMReLU).

In [ ]:
# Generate sequences for train and test sets
feature_columns = total_data.drop(columns=['projid', 'study', 'fu_year', 'cogdx', 
                                             'amyloid', 'gpath', 'tangles', 'niareagansc']).columns.tolist()
target_columns = ['gpath', 'tangles', 'niareagansc']

train_ids = total_data.projid.unique()
train_sequences = utils_BiLasTrue_ReLU.create_sequences(total_data, train_ids, feature_columns, target_columns)

test_ids = total_data.projid.unique()
test_sequences = utils_BiLasTrue_ReLU.create_sequences(total_data, test_ids, feature_columns, target_columns)

test_r2, train_loss, val_loss, lr_history = utils_BiLasTrue_ReLU.train_and_evaluate_model(
    train_data=train_sequences,
    test_data=test_sequences,
    input_dim=len(feature_columns),
    output_dim=len(target_columns),  
    num_epochs=500,
    patience=10,
    lr_scheduler_patience=5,
    lr_factor=0.5,
    hidden_size=8,
    num_layers=4,
    batch_size=16,
    learning_rate=0.01,
    seed=1217,
    temporary=True,
    dropout_rate=0,
    bidirectional=False
)

In [ ]:
test_r2_with_names = dict(zip(target_columns, [round(value, 6) for value in test_r2]))
print(test_r2_with_names)

- Using the best parameter for gpath, the LSTM test $R^2$ for gpath is 0.309175
- The elastic-net test $R^2$ for gpath is 0.262

In [ ]:
import numpy as np
epochs = np.arange(len(train_loss))
lowest_val_idx = np.argmin(val_loss)

plt.figure(figsize=(8, 6))
plt.plot(epochs, train_loss, label='Training Loss', marker='o')
plt.plot(epochs, val_loss, label='Validation Loss', marker='o')

# Annotate the lowest validation loss
plt.scatter(lowest_val_idx, val_loss[lowest_val_idx], color='red', zorder=3)
plt.text(lowest_val_idx + 1, val_loss[lowest_val_idx] - 0.05, 
         f'Saved Model at Epoch {lowest_val_idx}', 
         fontsize=10, verticalalignment='bottom', horizontalalignment='right')

plt.xticks(epochs)  # Set x-axis ticks explicitly as factors (0, 1, 2, ...)
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training and Validation Loss')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.6)
plt.show()

When you collect the "training loss" during each epoch, you're effectively averaging the loss across all mini‐batches, including the very first batches of that epoch when the model parameters are still relatively unoptimized (or only partially updated). By the time you finish the epoch, the model has been updated multiple times, so when you switch to evaluation mode (model.eval()) and compute the "validation loss," that loss is based on the final state of the model at the end of the epoch—i.e., a model that's already received a full epoch's worth of updates.

Hence, it's not unusual to see:

- Training Loss (Epoch Average): Higher (especially early batches) because it includes loss from the very start of the epoch.
- Validation Loss (After Epoch): Potentially lower because it uses the (better) model at the end of the epoch's training.

### Train the selected model on the entire dataset (combining both training and testing data) to predict estimated brain pathologies for subsequent clustering analysis.

In [ ]:
all_ids = total_data.projid.unique()
all_sequences = utils_BiLasTrue_ReLU.create_sequences(total_data, all_ids, feature_columns, target_columns)

training_r2, train_loss, val_loss, lr_history = utils_BiLasTrue_ReLU.train_and_evaluate_model(
    train_data=all_sequences,
    test_data=all_sequences,
    input_dim=len(feature_columns),
    output_dim=len(target_columns),  
    num_epochs=500,
    patience=10,
    lr_scheduler_patience=5,
    lr_factor=0.5,
    hidden_size=8,
    num_layers=4,
    batch_size=16,
    learning_rate=0.01,
    seed=1217,
    model_save_path="outputs/final_model_gpath_LSTMReLU.pth",
    dropout_rate=0,
    bidirectional=False
)

In [ ]:
training_r2_with_names = dict(zip(target_columns, [round(value, 4) for value in training_r2]))
print(training_r2_with_names)

Training $R^2$s look reasonable. 

# Final model for tangles (LSTM)

In [ ]:
# Generate sequences for train and test sets
feature_columns = total_data.drop(columns=['projid', 'study', 'fu_year', 'cogdx', 
                                             'amyloid', 'gpath', 'tangles', 'niareagansc']).columns.tolist()
target_columns = ['amyloid', 'tangles', 'niareagansc']

train_ids = total_data.projid.unique()
train_sequences = utils_BiLSTMasTrue.create_sequences(total_data, train_ids, feature_columns, target_columns)

test_ids = total_data.projid.unique()
test_sequences = utils_BiLSTMasTrue.create_sequences(total_data, test_ids, feature_columns, target_columns)

test_r2, train_loss, val_loss, lr_history = utils_BiLSTMasTrue.train_and_evaluate_model(
    train_data=train_sequences,
    test_data=test_sequences,
    input_dim=len(feature_columns),
    output_dim=len(target_columns),  
    num_epochs=500,
    patience=10,
    lr_scheduler_patience=5,
    lr_factor=0.5,
    hidden_size=16,
    num_layers=3,
    batch_size=64,
    learning_rate=0.01,
    seed=1217,
    temporary=True,
    dropout_rate=0,
    bidirectional=False
)

In [ ]:
test_r2_with_names = dict(zip(target_columns, [round(value, 6) for value in test_r2]))
print(test_r2_with_names)

In [ ]:
import numpy as np
epochs = np.arange(len(train_loss))
lowest_val_idx = np.argmin(val_loss)

plt.figure(figsize=(8, 6))
plt.plot(epochs, train_loss, label='Training Loss', marker='o')
plt.plot(epochs, val_loss, label='Validation Loss', marker='o')

# Annotate the lowest validation loss
plt.scatter(lowest_val_idx, val_loss[lowest_val_idx], color='red', zorder=3)
plt.text(lowest_val_idx + 1, val_loss[lowest_val_idx] - 0.05, 
         f'Saved Model at Epoch {lowest_val_idx}', 
         fontsize=10, verticalalignment='bottom', horizontalalignment='right')

plt.xticks(epochs)  # Set x-axis ticks explicitly as factors (0, 1, 2, ...)
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training and Validation Loss')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.6)
plt.show()

In [ ]:
all_ids = total_data.projid.unique()
all_sequences = utils_BiLSTMasTrue.create_sequences(total_data, all_ids, feature_columns, target_columns)

training_r2, train_loss, val_loss, lr_history = utils_BiLSTMasTrue.train_and_evaluate_model(
    train_data=all_sequences,
    test_data=all_sequences,
    input_dim=len(feature_columns),
    output_dim=len(target_columns),  
    num_epochs=500,
    patience=10,
    lr_scheduler_patience=5,
    lr_factor=0.5,
    hidden_size=16,
    num_layers=3,
    batch_size=64,
    learning_rate=0.01,
    seed=1217,
    model_save_path="outputs/final_model_tangles_LSTM.pth",
    dropout_rate=0,
    bidirectional=False
)

In [ ]:
training_r2_with_names = dict(zip(target_columns, [round(value, 6) for value in training_r2]))
print(training_r2_with_names)

# Final model for amyloid (BiLSTM)

In [ ]:
# Generate sequences for train and test sets
feature_columns = total_data.drop(columns=['projid', 'study', 'fu_year', 'cogdx', 
                                             'amyloid', 'gpath', 'tangles', 'niareagansc']).columns.tolist()
target_columns = ['gpath', 'tangles', 'amyloid', 'niareagansc']

train_ids = total_data.projid.unique()
train_sequences = utils_BiLSTMasTrue.create_sequences(total_data, train_ids, feature_columns, target_columns)

test_ids = total_data.projid.unique()
test_sequences = utils_BiLSTMasTrue.create_sequences(total_data, test_ids, feature_columns, target_columns)

test_r2, train_loss, val_loss, lr_history = utils_BiLSTMasTrue.train_and_evaluate_model(
    train_data=train_sequences,
    test_data=test_sequences,
    input_dim=len(feature_columns),
    output_dim=len(target_columns),  
    num_epochs=500,
    patience=10,
    lr_scheduler_patience=5,
    lr_factor=0.5,
    hidden_size=32,
    num_layers=3,
    batch_size=32,
    learning_rate=0.01,
    seed=1217,
    temporary=True,
    dropout_rate=0.2
)

In [ ]:
test_r2_with_names = dict(zip(target_columns, [round(value, 6) for value in test_r2]))
print(test_r2_with_names)

In [ ]:
import numpy as np
epochs = np.arange(len(train_loss))
lowest_val_idx = np.argmin(val_loss)

plt.figure(figsize=(8, 6))
plt.plot(epochs, train_loss, label='Training Loss', marker='o')
plt.plot(epochs, val_loss, label='Validation Loss', marker='o')

# Annotate the lowest validation loss
plt.scatter(lowest_val_idx, val_loss[lowest_val_idx], color='red', zorder=3)
plt.text(lowest_val_idx + 1, val_loss[lowest_val_idx] - 0.05, 
         f'Saved Model at Epoch {lowest_val_idx}', 
         fontsize=10, verticalalignment='bottom', horizontalalignment='right')

plt.xticks(epochs)  # Set x-axis ticks explicitly as factors (0, 1, 2, ...)
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training and Validation Loss')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.6)
plt.show()

In [ ]:
all_ids = total_data.projid.unique()
all_sequences = utils_BiLSTMasTrue.create_sequences(total_data, all_ids, feature_columns, target_columns)

training_r2, train_loss, val_loss, lr_history = utils_BiLSTMasTrue.train_and_evaluate_model(
    train_data=all_sequences,
    test_data=all_sequences,
    input_dim=len(feature_columns),
    output_dim=len(target_columns),  
    num_epochs=500,
    patience=10,
    lr_scheduler_patience=5,
    lr_factor=0.5,
    hidden_size=32,
    num_layers=3,
    batch_size=32,
    learning_rate=0.01,
    seed=1217,
    model_save_path="outputs/final_model_amyloid_BiLSTM.pth",
    dropout_rate=0.2
)

In [ ]:
training_r2_with_names = dict(zip(target_columns, [round(value, 6) for value in training_r2]))
print(training_r2_with_names)

# Final model for nia (BiLSTM)

In [ ]:
# Generate sequences for train and test sets
feature_columns = total_data.drop(columns=['projid', 'study', 'fu_year', 'cogdx', 
                                             'amyloid', 'gpath', 'tangles', 'niareagansc']).columns.tolist()
target_columns = ['gpath', 'tangles', 'amyloid', 'niareagansc']

train_ids = total_data.projid.unique()
train_sequences = utils_BiLSTMasTrue.create_sequences(total_data, train_ids, feature_columns, target_columns)

test_ids = total_data.projid.unique()
test_sequences = utils_BiLSTMasTrue.create_sequences(total_data, test_ids, feature_columns, target_columns)

test_r2, train_loss, val_loss, lr_history = utils_BiLSTMasTrue.train_and_evaluate_model(
    train_data=train_sequences,
    test_data=test_sequences,
    input_dim=len(feature_columns),
    output_dim=len(target_columns),  
    num_epochs=500,
    patience=10,
    lr_scheduler_patience=5,
    lr_factor=0.5,
    hidden_size=32,
    num_layers=4,
    batch_size=32,
    learning_rate=0.005,
    seed=1217,
    temporary=True,
    dropout_rate=0
)

In [ ]:
test_r2_with_names = dict(zip(target_columns, [round(value, 6) for value in test_r2]))
print(test_r2_with_names)

- Using the best parameter for nia, the LSTM test $R^2$ for tangles is 0.35061
- The elastic-net test $R^2$ for tangles is 0.305

In [ ]:
import numpy as np
epochs = np.arange(len(train_loss))
lowest_val_idx = np.argmin(val_loss)

plt.figure(figsize=(8, 6))
plt.plot(epochs, train_loss, label='Training Loss', marker='o')
plt.plot(epochs, val_loss, label='Validation Loss', marker='o')

# Annotate the lowest validation loss
plt.scatter(lowest_val_idx, val_loss[lowest_val_idx], color='red', zorder=3)
plt.text(lowest_val_idx + 1, val_loss[lowest_val_idx] - 0.05, 
         f'Saved Model at Epoch {lowest_val_idx}', 
         fontsize=10, verticalalignment='bottom', horizontalalignment='right')

plt.xticks(epochs)  # Set x-axis ticks explicitly as factors (0, 1, 2, ...)
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training and Validation Loss')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.6)
plt.show()

### Train the selected model on the entire dataset (combining both training and testing data) to predict estimated brain pathologies for subsequent clustering analysis.

In [ ]:
all_ids = total_data.projid.unique()
all_sequences = utils_BiLSTMasTrue.create_sequences(total_data, all_ids, feature_columns, target_columns)

training_r2, train_loss, val_loss, lr_history = utils_BiLSTMasTrue.train_and_evaluate_model(
    train_data=all_sequences,
    test_data=all_sequences,
    input_dim=len(feature_columns),
    output_dim=len(target_columns),  
    num_epochs=500,
    patience=10,
    lr_scheduler_patience=5,
    lr_factor=0.5,
    hidden_size=32,
    num_layers=4,
    batch_size=32,
    learning_rate=0.005,
    seed=1217,
    model_save_path="outputs/final_model_nia_BiLSTM.pth",
    dropout_rate=0
)

In [ ]:
training_r2_with_names = dict(zip(target_columns, [round(value, 6) for value in training_r2]))
print(training_r2_with_names)

Training $R^2$ looks reasonable.